<a href="https://colab.research.google.com/github/ACH-S-M/Nlp/blob/main/Salinan_dari_PERT_9_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LKM Pertemuan 9 — Perbaikan Model Text Mining & NLP**

- **Nama:** Muhamad Ihsan Mustaqim
- **NPM:** 202333500591
- **Kelas:** R7A
- **Mata Kuliah:** Business Intelligence
- **Dataset:** `tokopedia-product-reviews-2019.csv` (Kaggle — ~40.607 ulasan) & `colloquial-indonesian-lexicon.csv` (Google Drive)

Notebook ini disusun dalam dua bagian utama:
1. **Bagian 0 — Baseline (Sebelum Perbaikan):** Memuat dataset dan mereproduksi hasil awal (*Model Sentimen Malas* & *LDA Topik Sampah*) sebagai pembanding.
2. **Bagian Perbaikan (Tugas 1 s/d Tugas 4):** Normalisasi Slang, Pembersihan Lanjutan, Penanganan Imbalance Data, dan Perbaikan Topic Modeling (LDA) per sentimen.

---
# **Bagian 0: Load Dataset & Pipeline Baseline (Sebelum Perbaikan)**

Memuat dataset `tokopedia-product-reviews-2019.csv` dari Google Drive (`/content/drive/MyDrive/colab`). Jika file belum ada, otomatis diunduh dari Kaggle menggunakan `kagglehub`.

In [ ]:
# ------------------------------------------------------------------
# 1. Mount Google Drive & Load Dataset Tokopedia Product Reviews
# ------------------------------------------------------------------
import os
import shutil
import pandas as pd
import numpy as np
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

TARGET_DIR = '/content/drive/MyDrive/colab'
os.makedirs(TARGET_DIR, exist_ok=True)
print("Target folder:", TARGET_DIR)

DATASET_PATH = os.path.join(TARGET_DIR, 'tokopedia-product-reviews-2019.csv')

# Unduh dari Kaggle hanya jika file belum ada di Google Drive
if not os.path.exists(DATASET_PATH):
    !pip install -q --upgrade kagglehub
    import kagglehub
    dataset_cache_path = kagglehub.dataset_download("farhan999/tokopedia-product-reviews")
    print("Dataset cached at:", dataset_cache_path)

    for root, _, files in os.walk(dataset_cache_path):
        for f in files:
            src = os.path.join(root, f)
            dst = os.path.join(TARGET_DIR, f)
            if not os.path.exists(dst):
                shutil.copy2(src, dst)
                print(f"Copied: {f} -> {dst}")
else:
    print(f"File sudah ada di Drive: {DATASET_PATH}")

# Load dataset utama (jangan dimodifikasi langsung agar baseline & perbaikan terpisah)
df_raw = pd.read_csv(DATASET_PATH)
df_raw = df_raw.dropna(subset=['text']).reset_index(drop=True)

# Mapping rating -> sentimen (4-5: positive, 3: neutral, 1-2: negative)
df_raw['sentiment'] = df_raw['rating'].apply(
    lambda x: 'positive' if x >= 4 else ('neutral' if x == 3 else 'negative')
)

print(f"\nShape dataset: {df_raw.shape}")
print("\nDistribusi Sentimen:")
dist = pd.DataFrame({
    'Jumlah': df_raw['sentiment'].value_counts(),
    'Persentase (%)': (df_raw['sentiment'].value_counts(normalize=True) * 100).round(2)
})
display(dist)
display(df_raw[['text', 'rating', 'sentiment', 'category']].head())

In [ ]:
# ------------------------------------------------------------------
# 2. Install Library & Inisialisasi Sastrawi (dengan Caching agar Cepat)
# ------------------------------------------------------------------
!pip install -q Sastrawi nltk tqdm

import re
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from tqdm.auto import tqdm

nltk.download('punkt_tab', quiet=True)
tqdm.pandas()

# Stemmer Sastrawi + Cache (hasil 100% identik, namun 15-30x lebih cepat untuk 40k baris)
stemmer = StemmerFactory().create_stemmer()
stem_cache = {}

def fast_stem(word):
    if word not in stem_cache:
        stem_cache[word] = stemmer.stem(word)
    return stem_cache[word]

# Stopwords bawaan Sastrawi
stopword_factory = StopWordRemoverFactory()
stop_words_sastrawi = set(stopword_factory.get_stop_words())

print(f"Jumlah stopwords bawaan Sastrawi: {len(stop_words_sastrawi)} kata")

In [ ]:
# ------------------------------------------------------------------
# 3. Pipeline Baseline (Sebelum Perbaikan) — Sebagai Pembanding
# ------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.decomposition import LatentDirichletAllocation

def preprocess_baseline(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words_sastrawi and len(w) > 2]
    tokens = [fast_stem(w) for w in tokens]
    return ' '.join(tokens)

# Variabel dataset sebelum perbaikan: df_baseline
df_baseline = df_raw.copy()
print("Menjalankan preprocessing baseline pada 40k ulasan...")
df_baseline['clean'] = df_baseline['text'].progress_apply(preprocess_baseline)

# TF-IDF Baseline
tfidf_base = TfidfVectorizer(max_features=1000, min_df=5, max_df=0.8)
X_base = tfidf_base.fit_transform(df_baseline['clean'])

features_base = tfidf_base.get_feature_names_out()
avg_base = np.asarray(X_base.mean(axis=0)).flatten()
top20_base = [features_base[i] for i in avg_base.argsort()[-20:][::-1]]
print(f"\nShape TF-IDF Baseline: {X_base.shape}")
print("Top 20 kata Baseline:", top20_base)

# Sentimen Naive Bayes Baseline (tanpa penanganan imbalance)
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_base, df_baseline['sentiment'], test_size=0.2, random_state=42,
    stratify=df_baseline['sentiment']
)

model_base = MultinomialNB()
model_base.fit(X_train_b, y_train_b)
y_pred_b = model_base.predict(X_test_b)

print("\n=== CLASSIFICATION REPORT: MODEL BASELINE (SEBELUM PERBAIKAN) ===")
print(classification_report(y_test_b, y_pred_b))
report_base = classification_report(y_test_b, y_pred_b, output_dict=True)

# LDA Baseline (seluruh data digabung)
lda_base = LatentDirichletAllocation(n_components=5, random_state=42)
lda_base.fit(X_base)

print("=== LDA BASELINE (SELURUH DATA) ===")
for i, topic in enumerate(lda_base.components_):
    top_words = [features_base[j] for j in topic.argsort()[-10:][::-1]]
    print(f"Topik {i+1}: {' | '.join(top_words)}")

---
# **TUGAS 1: NORMALISASI KATA SLANG**

**Tujuan:** Mengubah bahasa gaul/singkatan (*slang*) menjadi kata baku menggunakan kamus *open-source* `colloquial-indonesian-lexicon.csv` yang tersimpan di Google Drive (`/content/drive/MyDrive/colab/colloquial-indonesian-lexicon.csv`), dilengkapi penyesuaian istilah e-commerce Tokopedia.

In [ ]:
# ------------------------------------------------------------------
# TUGAS 1: NORMALISASI KATA SLANG
# ------------------------------------------------------------------
SLANG_PATH = os.path.join(TARGET_DIR, 'colloquial-indonesian-lexicon.csv')

# Load kamus alay dari Google Drive (dengan fallback ke GitHub jika belum ada di Drive)
if os.path.exists(SLANG_PATH):
    slang_df = pd.read_csv(SLANG_PATH)
    print(f"Kamus slang dimuat dari Google Drive: {SLANG_PATH}")
else:
    SLANG_URL = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
    slang_df = pd.read_csv(SLANG_URL)
    slang_df.to_csv(SLANG_PATH, index=False)
    print(f"Kamus slang diunduh dari GitHub dan disimpan ke: {SLANG_PATH}")

# Bersihkan & prioritaskan kata formal yang terdaftar di kamus baku (In-dictionary == 1)
slang_df = slang_df.dropna(subset=['slang', 'formal']).copy()
slang_df['slang'] = slang_df['slang'].astype(str).str.strip().str.lower()
slang_df['formal'] = slang_df['formal'].astype(str).str.strip().str.lower()

if 'In-dictionary' in slang_df.columns:
    slang_df = slang_df.sort_values(by='In-dictionary', ascending=False)

slang_df = slang_df.drop_duplicates(subset='slang', keep='first')
slang_dict = dict(zip(slang_df['slang'], slang_df['formal']))

# Penyesuaian & tambahan slang khusus konteks e-commerce Tokopedia
# Catatan: Di kamus Twitter 'brg' -> 'bareng', padahal di ulasan Tokopedia 'brg' berarti 'barang'.
# Variasi negasi ('ga', 'gak', 'gk', 'enggak', dll.) disatukan menjadi 'tidak' agar konsisten.
ecommerce_slang = {
    'brg': 'barang',
    'brng': 'barang',
    'barangnya': 'barang',
    'barangya': 'barang',
    'paking': 'packing',
    'pakingnya': 'packing',
    'packingnya': 'packing',
    'pengirimanya': 'pengiriman',
    'pengirimannya': 'pengiriman',
    'responya': 'respon',
    'responnya': 'respon',
    'ga': 'tidak',
    'gak': 'tidak',
    'gk': 'tidak',
    'ngga': 'tidak',
    'nggak': 'tidak',
    'enggak': 'tidak',
    'tdk': 'tidak',
    'tak': 'tidak',
    'g': 'tidak',
    'bgt': 'banget',
    'bngt': 'banget',
    'cepet': 'cepat',
    'cpt': 'cepat',
    'sampe': 'sampai',
    'nyampe': 'sampai',
    'dgn': 'dengan',
    'dg': 'dengan',
    'yg': 'yang',
    'udh': 'sudah',
    'udah': 'sudah',
    'blm': 'belum',
    'belom': 'belum',
    'krn': 'karena',
    'karna': 'karena',
    'tp': 'tapi',
    'tpi': 'tapi',
    'sy': 'saya',
    'bkn': 'bukan',
    'bgs': 'bagus',
    'mksh': 'terima kasih',
    'makasih': 'terima kasih',
    'trims': 'terima kasih',
    'thx': 'terima kasih',
    'tq': 'terima kasih',
    'tks': 'terima kasih',
    'mantul': 'mantap',
    'mantab': 'mantap',
    'ori': 'original',
    'rekomen': 'rekomendasi',
    'recomended': 'rekomendasi',
    'recommended': 'rekomendasi',
    'kecewaa': 'kecewa',
    'lamaa': 'lama',
    'parahh': 'parah',
}
slang_dict.update(ecommerce_slang)

print(f"Total entri kamus slang siap pakai: {len(slang_dict):,} kata")

# Fungsi untuk memproses kata slang dalam sebuah kalimat
def normalisasi_slang(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = word_tokenize(text)
    normalized_tokens = []
    for w in tokens:
        kata_baku = slang_dict.get(w, w)
        # Split ulang jika kata baku terdiri dari >1 kata (misal: 'mksh' -> 'terima kasih')
        normalized_tokens.extend(kata_baku.split())
    return ' '.join(normalized_tokens)

# Uji coba fungsi normalisasi slang pada kalimat contoh
contoh_kalimat = [
    "Brg bagus bgt!!! Cepet sampe, paking aman dgn harga murah, tq gan",
    "Kecewa bgt brg gak sesuai pesanan, pengirimanya lama bngt, blm dipake udah rusak",
]

print("\n=== UJI COBA NORMALISASI KATA SLANG ===")
for idx, kal in enumerate(contoh_kalimat, 1):
    print(f"Contoh {idx}")
    print(f"  Sebelum : {kal}")
    print(f"  Sesudah : {normalisasi_slang(kal)}\n")

---
# **TUGAS 2: PEMBERSIHAN LANJUTAN**

**Tujuan:**
1. Membuat daftar *custom stopwords* berisi kata-kata sampah/sapaan/formalitas yang sebelumnya mendominasi LDA awal (`terima`, `kasih`, `makasih`, `gan`, `mantap`, `bintang`, `sukses`, dll.).
2. Menggabungkan *custom stopwords* tersebut dengan *stopwords* bawaan Sastrawi — **sambil mempertahankan kata negasi** (`tidak`, `bukan`, `belum`, `jangan`, `kurang`, `tanpa`) agar keluhan pelanggan tidak kehilangan makna.
3. Menerapkan fungsi preprocessing lengkap ke variabel dataset baru (`df_fix`, kolom `clean_fix`) agar dapat dibandingkan langsung dengan dataset baseline (`df_baseline`, kolom `clean`).

In [ ]:
# ------------------------------------------------------------------
# TUGAS 2: PEMBERSIHAN LANJUTAN
# ------------------------------------------------------------------

# 1. Custom stopwords: kata formalitas, sapaan, filler, dan kata sampah dari LDA awal
custom_stopwords = {
    # Ucapan terima kasih & sapaan
    'terima', 'kasih', 'makasih', 'terimakasih', 'trims', 'thanks', 'thank',
    'thankyou', 'thx', 'tks', 'tq', 'gan', 'agan', 'bro', 'sis', 'bos', 'boss',
    'kak', 'kaka', 'kakak', 'min', 'bang', 'mas', 'mba', 'mbak', 'om', 'tante',
    # Kata pujian/formalitas generik tanpa wawasan bisnis spesifik
    'mantap', 'mantab', 'mantul', 'bintang', 'sukses', 'selamat', 'selalu',
    'bicara', 'pokoknya', 'semoga', 'berkah', 'laris', 'manis', 'lancar',
    'jaya', 'terus', 'pertahankan', 'tingkatkan', 'alhamdulillah', 'hehe',
    'haha', 'wkwk', 'oke', 'okey', 'ok', 'sip', 'sipp', 'top', 'jos', 'joss',
    'good', 'nice', 'best', 'very', 'great', 'the', 'and', 'for', 'with',
    # Partikel percakapan & kata pengisi
    'nya', 'dah', 'deh', 'sih', 'dong', 'nih', 'kok', 'kan', 'loh', 'lho',
    'tuh', 'aja', 'saja', 'gitu', 'kayak', 'banget', 'sangat', 'sekali',
    'sudah', 'udah', 'kali', 'pertama', 'langsung', 'sampe', 'nyaa',
}

# 2. Gabungkan dengan stopwords bawaan Sastrawi
# Penting: Keluarkan kata negasi agar frasa keluhan ("tidak sesuai", "belum sampai", "kurang bagus")
# tidak terhapus dan tetap bisa dideteksi oleh model sentimen & LDA negatif!
kata_negasi = {'tidak', 'bukan', 'belum', 'jangan', 'kurang', 'tanpa'}
stop_words_fix = (stop_words_sastrawi | custom_stopwords) - kata_negasi

print(f"Stopwords Sastrawi       : {len(stop_words_sastrawi)} kata")
print(f"Custom Stopwords         : {len(custom_stopwords)} kata")
print(f"Kata Negasi Dipertahankan: {sorted(kata_negasi)}")
print(f"Total Stopwords Gabungan : {len(stop_words_fix)} kata")

# 3. Fungsi preprocessing lengkap
def preprocess_fix(text):
    if not isinstance(text, str):
        return ''
    # (a) Case folding
    text = text.lower()
    # (b) Hapus tanda baca & angka
    text = re.sub(r'[^a-z\s]', ' ', text)
    # (c) Tokenisasi
    tokens = word_tokenize(text)
    # (d) Normalisasi slang (dari Tugas 1, dengan pemecahan multi-kata)
    normalized = []
    for w in tokens:
        kata_baku = slang_dict.get(w, w)
        normalized.extend(kata_baku.split())
    # (e) Hapus stopwords gabungan & kata terlalu pendek (<= 2 huruf)
    filtered = [w for w in normalized if w not in stop_words_fix and len(w) > 2]
    # (f) Stemming Sastrawi (dengan cache agar cepat)
    stemmed = [fast_stem(w) for w in filtered]
    # (g) Bersihkan ulang jika hasil stemming berubah menjadi stopword
    final_tokens = [w for w in stemmed if w not in stop_words_fix and len(w) > 2]
    return ' '.join(final_tokens)

# Uji cepat sebelum dijalankan ke seluruh dataset
sample_uji = "Makasih gan brg bagus bgt, mantap dah! Tp pengirimanya agak lama & packing gak rapi."
print("\n=== UJI FUNGSI PREPROCESS_FIX ===")
print("Teks Asli       :", sample_uji)
print("Hasil clean_fix :", preprocess_fix(sample_uji))

# 4. Terapkan ke dataset BARU (df_fix) dan simpan di kolom baru (clean_fix)
df_fix = df_raw.copy()
print("\nMenjalankan preprocessing lanjutan pada 40k ulasan (df_fix)...")
df_fix['clean_fix'] = df_fix['text'].progress_apply(preprocess_fix)

# Simpan backup ke Google Drive agar tidak perlu mengulang stemming jika runtime restart
CLEAN_FIX_PATH = os.path.join(TARGET_DIR, 'tokopedia_reviews_clean_fix.csv')
df_fix.to_csv(CLEAN_FIX_PATH, index=False)
print(f"Hasil bersih disimpan ke: {CLEAN_FIX_PATH}")

# Tampilkan perbandingan potongan hasil Sebelum (clean) vs Sesudah (clean_fix)
perbandingan_cols = {
    'Teks Asli (text)': df_fix['text'],
    'Sesudah Perbaikan (clean_fix)': df_fix['clean_fix'],
    'Sentiment': df_fix['sentiment'],
}
if 'df_baseline' in globals() and 'clean' in df_baseline.columns:
    perbandingan_cols = {
        'Teks Asli (text)': df_fix['text'],
        'Sebelum Perbaikan (clean)': df_baseline['clean'],
        'Sesudah Perbaikan (clean_fix)': df_fix['clean_fix'],
        'Sentiment': df_fix['sentiment'],
    }
perbandingan_clean = pd.DataFrame(perbandingan_cols)

print("\n=== PERBANDINGAN HASIL PREPROCESSING (10 SAMPEL AWAL) ===")
display(perbandingan_clean.head(10))

print("\n=== PERBANDINGAN HASIL PREPROCESSING (5 SAMPEL REVIEW NEGATIF) ===")
display(perbandingan_clean[perbandingan_clean['Sentiment'] == 'negative'].head(5))

---
# **TUGAS 3: PENANGANAN IMBALANCE DATA**

**Tujuan:** Memaksa model Naive Bayes lebih peduli pada kelas minoritas (`negative` dan `neutral`) agar mampu mendeteksi keluhan pelanggan.

> **Catatan Teknis `scikit-learn`:**  
> Pada `scikit-learn`, konstruktor `MultinomialNB()` **tidak** memiliki parameter `class_weight='balanced'` (memanggil `MultinomialNB(class_weight='balanced')` langsung akan menimbulkan `TypeError`). Cara resmi di `scikit-learn` untuk menerapkan `class_weight='balanced'` pada `MultinomialNB` adalah menghitung bobot sampel seimbang menggunakan `compute_sample_weight(class_weight='balanced', y=y_train)` lalu meneruskannya ke `model.fit(..., sample_weight=sample_weights)`.

In [ ]:
# ------------------------------------------------------------------
# TUGAS 3: PENANGANAN IMBALANCE DATA
# ------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix

# 1. Buat matriks TF-IDF baru menggunakan kolom clean_fix
# Menggunakan ngram_range=(1, 2) agar frasa negasi seperti "tidak sesuai", "tidak fungsi",
# "kirim lama" tertangkap sebagai fitur penting
tfidf_fix = TfidfVectorizer(
    max_features=3000,
    min_df=5,
    max_df=0.8,
    ngram_range=(1, 2)
)
X_fix = tfidf_fix.fit_transform(df_fix['clean_fix'])

features_fix = tfidf_fix.get_feature_names_out()
avg_fix = np.asarray(X_fix.mean(axis=0)).flatten()
top20_fix = [features_fix[i] for i in avg_fix.argsort()[-20:][::-1]]

print(f"Shape TF-IDF Baru (X_fix): {X_fix.shape}")
print("Top 20 fitur baru (clean_fix):", top20_fix)

# 2. Train-Test Split 80% latih, 20% uji dengan stratify
X_train_fix, X_test_fix, y_train_fix, y_test_fix = train_test_split(
    X_fix, df_fix['sentiment'], test_size=0.2, random_state=42,
    stratify=df_fix['sentiment']
)

# 3. Latih Multinomial Naive Bayes dengan pembobotan kelas seimbang (class_weight='balanced')
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_fix)

model_fix = MultinomialNB()
model_fix.fit(X_train_fix, y_train_fix, sample_weight=sample_weights)

# 4. Prediksi & Cetak Classification Report
y_pred_fix = model_fix.predict(X_test_fix)

print("\n=== CLASSIFICATION REPORT: MODEL BARU (MULTINOMIAL NB + BALANCED WEIGHT) ===")
print(classification_report(y_test_fix, y_pred_fix))
report_fix = classification_report(y_test_fix, y_pred_fix, output_dict=True)

# 5. Tabel Perbandingan Model Awal vs Model Baru (jika baseline sudah dijalankan)
if 'report_base' in globals():
    tabel_banding = pd.DataFrame({
        'Metrik': [
            'Accuracy',
            'Recall (Negative)',
            'Precision (Negative)',
            'F1-Score (Negative)',
            'Recall (Neutral)',
            'Macro Avg F1-Score',
        ],
        'Model Awal (Baseline)': [
            round(report_base['accuracy'], 4),
            round(report_base['negative']['recall'], 4),
            round(report_base['negative']['precision'], 4),
            round(report_base['negative']['f1-score'], 4),
            round(report_base['neutral']['recall'], 4),
            round(report_base['macro avg']['f1-score'], 4),
        ],
        'Model Baru (Balanced)': [
            round(report_fix['accuracy'], 4),
            round(report_fix['negative']['recall'], 4),
            round(report_fix['negative']['precision'], 4),
            round(report_fix['negative']['f1-score'], 4),
            round(report_fix['neutral']['recall'], 4),
            round(report_fix['macro avg']['f1-score'], 4),
        ],
    })
    tabel_banding['Perubahan (Delta)'] = (
        tabel_banding['Model Baru (Balanced)'] - tabel_banding['Model Awal (Baseline)']
    ).round(4)

    print("=== PERBANDINGAN EVALUASI: SEBELUM vs SESUDAH PERBAIKAN ===")
    display(tabel_banding)

    # 6. Visualisasi Confusion Matrix (Baseline vs Model Baru)
    labels = ['negative', 'neutral', 'positive']
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    cm_base = confusion_matrix(y_test_b, y_pred_b, labels=labels)
    sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=axes[0])
    axes[0].set_title('Confusion Matrix — Model Awal (Baseline)')
    axes[0].set_xlabel('Prediksi')
    axes[0].set_ylabel('Aktual')

    cm_fix = confusion_matrix(y_test_fix, y_pred_fix, labels=labels)
    sns.heatmap(cm_fix, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels, ax=axes[1])
    axes[1].set_title('Confusion Matrix — Model Baru (Balanced)')
    axes[1].set_xlabel('Prediksi')
    axes[1].set_ylabel('Aktual')

    plt.tight_layout()
    plt.show()
else:
    labels = ['negative', 'neutral', 'positive']
    plt.figure(figsize=(6, 5))
    cm_fix = confusion_matrix(y_test_fix, y_pred_fix, labels=labels)
    sns.heatmap(cm_fix, annot=True, fmt='d', cmap='Greens', xticklabels=labels, yticklabels=labels)
    plt.title('Confusion Matrix — Model Baru (Balanced)')
    plt.xlabel('Prediksi')
    plt.ylabel('Aktual')
    plt.tight_layout()
    plt.show()

---
# **TUGAS 4: PERBAIKAN TOPIC MODELING (LDA)**

**Tujuan:** Memisahkan dataset berdasarkan sentimen (`positive` dan `negative`), lalu menjalankan LDA (`n_components=3`) pada masing-masing subset untuk menemukan:
- **Keunggulan toko** dari ulasan `positive`
- **Keluhan utama pelanggan** dari ulasan `negative`

In [ ]:
# ------------------------------------------------------------------
# TUGAS 4: PERBAIKAN TOPIC MODELING (LDA) PER SENTIMEN
# ------------------------------------------------------------------
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Kata terlalu umum di seluruh ulasan e-commerce yang tidak membedakan antar topik bisnis
lda_extra_stopwords = ['barang', 'produk', 'tokopedia', 'beli']

hasil_lda = []

for sentimen in ['positive', 'negative']:
    print("=" * 75)
    print(f"TOPIC MODELING (LDA) — SENTIMEN: {sentimen.upper()}")
    print("=" * 75)

    # 1. Filter dataset berdasarkan sentimen
    df_sub = df_fix[df_fix['sentiment'] == sentimen].copy()
    # Pastikan hanya dokumen yang tidak kosong setelah cleaning
    df_sub = df_sub[df_sub['clean_fix'].str.strip() != '']
    print(f"Jumlah dokumen '{sentimen}': {len(df_sub):,}")

    # 2. Buat matriks TF-IDF khusus subset sentimen tersebut
    min_df_sub = 5 if len(df_sub) > 2000 else 3
    tfidf_sub = TfidfVectorizer(
        max_features=1000,
        min_df=min_df_sub,
        max_df=0.6,
        ngram_range=(1, 2),
        stop_words=lda_extra_stopwords
    )
    X_sub = tfidf_sub.fit_transform(df_sub['clean_fix'])
    features_sub = tfidf_sub.get_feature_names_out()

    # 3. Latih model LDA dengan n_components=3
    lda_sub = LatentDirichletAllocation(
        n_components=3,
        max_iter=20,
        learning_method='batch',
        random_state=42
    )
    lda_sub.fit(X_sub)

    # 4. Ekstrak dan cetak 10 kata dominan untuk setiap topik
    for i, topic in enumerate(lda_sub.components_, start=1):
        top_10 = [features_sub[j] for j in topic.argsort()[-10:][::-1]]
        kata_str = ' | '.join(top_10)
        print(f"Topik {i}: {kata_str}")
        hasil_lda.append({
            'Sentimen': sentimen.capitalize(),
            'Topik': i,
            '10 Kata Dominan': ', '.join(top_10)
        })
    print()

# Tampilkan ringkasan tabel agar mudah disalin ke LKM
df_hasil_lda = pd.DataFrame(hasil_lda)
print("=== RINGKASAN TABEL HASIL LDA (UNTUK LKM) ===")
pd.set_option('display.max_colwidth', None)
display(df_hasil_lda)